In [ ]:
!pip install pyyaml==5.4.1
!pip install darts
import darts
print(darts.__version__)

!pip install -U optuna==2.0.0

import numpy as np
import time

from darts import TimeSeries
from darts.utils.timeseries_generation import gaussian_timeseries, linear_timeseries, sine_timeseries
from darts.models import LightGBMModel, CatBoostModel, Prophet, RNNModel, TFTModel, NaiveSeasonal, ExponentialSmoothing, NHiTSModel
from darts.metrics import mape, smape, rmse, rmsle
from darts.dataprocessing import Pipeline
from darts.dataprocessing.transformers import Scaler, StaticCovariatesTransformer, MissingValuesFiller, InvertibleMapper
from darts.utils.timeseries_generation import datetime_attribute_timeseries
from darts.utils.statistics import check_seasonality, plot_acf, plot_residuals_analysis, plot_hist
from darts.utils.likelihood_models import QuantileRegression
from darts.utils.missing_values import fill_missing_values
from darts.models import MovingAverage

import optuna
from optuna.integration import PyTorchLightningPruningCallback
from optuna.visualization import (
    plot_optimization_history,
    plot_contour,
    plot_param_importances,
)

from pytorch_lightning.callbacks.early_stopping import EarlyStopping

from tqdm import tqdm

import sklearn
from sklearn import preprocessing

import pandas as pd
import torch
import matplotlib.pyplot as plt
import gc

%matplotlib inline
torch.manual_seed(1); np.random.seed(1)  # for reproducibility

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
0.22.0
Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


1. Preprocess data

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Load all Datasets
df_train = pd.read_csv('/content/drive/My Drive/Colab Notebooks/store-sales-time-series-forecasting/train.csv')
df_test = pd.read_csv('/content/drive/My Drive/Colab Notebooks/store-sales-time-series-forecasting/test.csv')
df_holidays_events = pd.read_csv('/content/drive/My Drive/Colab Notebooks/store-sales-time-series-forecasting/holidays_events.csv')
df_oil = pd.read_csv('/content/drive/My Drive/Colab Notebooks/store-sales-time-series-forecasting/oil.csv')
df_stores = pd.read_csv('/content/drive/My Drive/Colab Notebooks/store-sales-time-series-forecasting/stores.csv')
df_transactions = pd.read_csv('/content/drive/My Drive/Colab Notebooks/store-sales-time-series-forecasting/transactions.csv')
df_sample_submission = pd.read_csv('/content/drive/My Drive/Colab Notebooks/store-sales-time-series-forecasting/sample_submission.csv')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

# Merge train.csv with store.csv for future use

family_list = df_train['family'].unique()
store_list = df_stores['store_nbr'].unique()

train_merged = pd.merge(df_train, df_stores, on ='store_nbr')
train_merged = train_merged.sort_values(["store_nbr","family","date"])
train_merged = train_merged.astype({"store_nbr":'str', "family":'str', "city":'str',
                          "state":'str', "type":'str', "cluster":'str'})

# drop 'onpromotion' column of test.csv and sort it by 'store_nbr' and 'family' for prediction use

df_test_dropped = df_test.drop(['onpromotion'], axis=1)
df_test_sorted = df_test_dropped.sort_values(by=['store_nbr','family'])

train_merged.head(5)
# df_test_sorted.head(5)

,id,date,store_nbr,family
0,3000888,2017-08-16,1,AUTOMOTIVE
1782,3002670,2017-08-17,1,AUTOMOTIVE
3564,3004452,2017-08-18,1,AUTOMOTIVE
5346,3006234,2017-08-19,1,AUTOMOTIVE
7128,3008016,2017-08-20,1,AUTOMOTIVE


In [ ]:

# Create TimeSeries using Darts and arrange in a dict <k,v> = <family,TimeSeries for each store>, total 33 x 54 = 1782

family_TS_dict = {}

for family in family_list:
  df_family = train_merged.loc[train_merged['family'] == family]

  list_of_TS_family = TimeSeries.from_group_dataframe(
                                df_family,
                                time_col="date",
                                group_cols=["store_nbr","family"],  # individual time series are extracted by grouping `df` by `group_cols`
                                static_cols=["city","state","type","cluster"], # also extract these additional columns as static covariates
                                value_cols="sales", # target variable
                                fill_missing_dates=True,
                                freq='D')
  for ts in list_of_TS_family:
            ts = ts.astype(np.float32)

  list_of_TS_family = sorted(list_of_TS_family, key=lambda ts: int(ts.static_covariates_values()[0,0]))
  family_TS_dict[family] = list_of_TS_family

# Transform the Sales Data

family_pipeline_dict = {}
family_TS_transformed_dict = {}

for key in family_TS_dict:
  train_filler = MissingValuesFiller(verbose=False, n_jobs=-1, name="Fill NAs")
  static_cov_transformer = StaticCovariatesTransformer(verbose=False, transformer_cat = sklearn.preprocessing.OneHotEncoder(), name="Encoder") #OneHotEncoder would be better but takes longer
  log_transformer = InvertibleMapper(np.log1p, np.expm1, verbose=False, n_jobs=-1, name="Log-Transform")   
  train_scaler = Scaler(verbose=False, n_jobs=-1, name="Scaling")

  train_pipeline = Pipeline([train_filler,
                             static_cov_transformer,
                             log_transformer,
                             train_scaler])
     
  training_transformed = train_pipeline.fit_transform(family_TS_dict[key])
  family_pipeline_dict[key] = train_pipeline
  family_TS_transformed_dict[key] = training_transformed

# family_TS_transformed_dict['AUTOMOTIVE'][0].static_covariates.iloc[0,54:]
family_TS_transformed_dict['AUTOMOTIVE'][0]


static_covariates
family_AUTOMOTIVE    1.0
city_Ambato          0.0
city_Babahoyo        0.0
city_Cayambe         0.0
city_Cuenca          0.0
                    ... 
cluster_5            0.0
cluster_6            0.0
cluster_7            0.0
cluster_8            0.0
cluster_9            0.0
Name: sales, Length: 61, dtype: float64

In [ ]:

# Create 7-day and 28-day moving average of sales

sales_moving_average_7 = MovingAverage(window=7)
sales_moving_average_28 = MovingAverage(window=28)

sales_moving_averages_dict = {}

for key in family_TS_transformed_dict:
  sales_mas_family = []
  
  for ts in family_TS_transformed_dict[key]:
    ma_7 = sales_moving_average_7.filter(ts)
    ma_7 = TimeSeries.from_series(ma_7.pd_series())  
    ma_7 = ma_7.astype(np.float32)
    ma_7 = ma_7.with_columns_renamed(col_names=ma_7.components, col_names_new="sales_ma_7")
    ma_28 = sales_moving_average_28.filter(ts)
    ma_28 = TimeSeries.from_series(ma_28.pd_series())  
    ma_28 = ma_28.astype(np.float32)
    ma_28 = ma_28.with_columns_renamed(col_names=ma_28.components, col_names_new="sales_ma_28")
    mas = ma_7.stack(ma_28)
    sales_mas_family.append(mas)
  
  sales_moving_averages_dict[key] = sales_mas_family  

sales_moving_averages_dict["AUTOMOTIVE"][0]

<TimeSeries (DataArray) (date: 1688, component: 2, sample: 1)>
array([[[0.32305965],
        [0.3514414 ]],

       [[0.37806854],
        [0.3434372 ]],

       [[0.3761781 ],
        [0.3364335 ]],

       ...,

       [[0.44159347],
        [0.53641903]],

       [[0.3761878 ],
        [0.5554841 ]],

       [[0.4123902 ],
        [0.5436196 ]]], dtype=float32)
Coordinates:
  * date       (date) datetime64[ns] 2013-01-01 2013-01-02 ... 2017-08-15
  * component  (component) object 'sales_ma_7' 'sales_ma_28'
Dimensions without coordinates: sample
Attributes:
    static_covariates:  None
    hierarchy:          None

In [ ]:
   
# General Covariates (Time-Based and Oil)

full_time_period = pd.date_range(start='2013-01-01', end='2017-08-31', freq='D')

# Time-Based Covariates

year = datetime_attribute_timeseries(time_index = full_time_period, attribute="year")
month = datetime_attribute_timeseries(time_index = full_time_period, attribute="month")
day = datetime_attribute_timeseries(time_index = full_time_period, attribute="day")
dayofyear = datetime_attribute_timeseries(time_index = full_time_period, attribute="dayofyear")
weekday = datetime_attribute_timeseries(time_index = full_time_period, attribute="dayofweek")
weekofyear = datetime_attribute_timeseries(time_index = full_time_period, attribute="weekofyear")
timesteps = TimeSeries.from_times_and_values(times=full_time_period,
                                             values=np.arange(len(full_time_period)),
                                             columns=["linear_increase"])

time_cov = year.stack(month).stack(day).stack(dayofyear).stack(weekday).stack(weekofyear).stack(timesteps)
time_cov = time_cov.astype(np.float32)

# Transform
time_cov_scaler = Scaler(verbose=False, n_jobs=-1, name="Scaler")
time_cov_train, time_cov_val = time_cov.split_before(pd.Timestamp('20170816'))
time_cov_scaler.fit(time_cov_train) # ? only scale train data
time_cov_transformed = time_cov_scaler.transform(time_cov)

# time_cov_transformed[-50:].plot()
time_cov_transformed

<TimeSeries (DataArray) (time: 1704, component: 7, sample: 1)>
array([[[0.0000000e+00],
        [0.0000000e+00],
        [0.0000000e+00],
        ...,
        [1.6666667e-01],
        [0.0000000e+00],
        [0.0000000e+00]],

       [[0.0000000e+00],
        [0.0000000e+00],
        [3.3333335e-02],
        ...,
        [3.3333334e-01],
        [0.0000000e+00],
        [5.9276825e-04]],

       [[0.0000000e+00],
        [0.0000000e+00],
        [6.6666678e-02],
        ...,
...
        ...,
        [1.6666667e-01],
        [6.5384614e-01],
        [1.0082988e+00]],

       [[1.0000000e+00],
        [6.3636363e-01],
        [9.6666664e-01],
        ...,
        [3.3333334e-01],
        [6.5384614e-01],
        [1.0088916e+00]],

       [[1.0000000e+00],
        [6.3636363e-01],
        [1.0000001e+00],
        ...,
        [5.0000000e-01],
        [6.5384614e-01],
        [1.0094843e+00]]], dtype=float32)
Coordinates:
  * time       (time) datetime64[ns] 2013-01-01 2013-01-02 ... 2017-08-31
  * component  (component) object 'year' 'month' ... 'linear_increase'
Dimensions without coordinates: sample
Attributes:
    static_covariates:  None
    hierarchy:          None

In [ ]:

# Oil Price

oil = TimeSeries.from_dataframe(df_oil, 
                                time_col = 'date', 
                                value_cols = ['dcoilwtico'],
                                freq = 'D')

oil = oil.astype(np.float32)

# Transform
oil_filler = MissingValuesFiller(verbose=False, n_jobs=-1, name="Filler")
oil_scaler = Scaler(verbose=False, n_jobs=-1, name="Scaler")
oil_pipeline = Pipeline([oil_filler, oil_scaler])
oil_transformed = oil_pipeline.fit_transform(oil)

# Moving Averages for Oil Price
oil_moving_average_7 = MovingAverage(window=7)
oil_moving_average_28 = MovingAverage(window=28)

oil_moving_averages = []

ma_7 = oil_moving_average_7.filter(oil_transformed).astype(np.float32)
ma_7 = ma_7.with_columns_renamed(col_names=ma_7.components, col_names_new="oil_ma_7")
ma_28 = oil_moving_average_28.filter(oil_transformed).astype(np.float32)
ma_28 = ma_28.with_columns_renamed(col_names=ma_28.components, col_names_new="oil_ma_28")
oil_moving_averages = ma_7.stack(ma_28)

# Stack General Covariates Together

general_covariates = time_cov_transformed.stack(oil_transformed).stack(oil_moving_averages)

general_covariates

<TimeSeries (DataArray) (time: 1704, component: 10, sample: 1)>
array([[[0.        ],
        [0.        ],
        [0.        ],
        ...,
        [0.7929646 ],
        [0.79240197],
        [0.796154  ]],

       [[0.        ],
        [0.        ],
        [0.03333334],
        ...,
        [0.7929646 ],
        [0.7925303 ],
        [0.7960361 ]],

       [[0.        ],
        [0.        ],
        [0.06666668],
        ...,
...
        ...,
        [0.2400805 ],
        [0.24278495],
        [0.25032222]],

       [[1.        ],
        [0.6363636 ],
        [0.96666664],
        ...,
        [0.23415849],
        [0.24149394],
        [0.25014064]],

       [[1.        ],
        [0.6363636 ],
        [1.0000001 ],
        ...,
        [0.2495558 ],
        [0.24079117],
        [0.25054285]]], dtype=float32)
Coordinates:
  * time       (time) datetime64[ns] 2013-01-01 2013-01-02 ... 2017-08-31
  * component  (component) object 'year' 'month' ... 'oil_ma_7' 'oil_ma_28'
Dimensions without coordinates: sample
Attributes:
    static_covariates:  None
    hierarchy:          None

In [ ]:
# Store-Specific Covariates (Transactions and Holidays)

# Transactions
df_transactions.sort_values(["store_nbr","date"], inplace=True)

TS_transactions_list = TimeSeries.from_group_dataframe(
                                df_transactions,
                                time_col="date",
                                group_cols=["store_nbr"],  # individual time series are extracted by grouping `df` by `group_cols`
                                value_cols="transactions",
                                fill_missing_dates=True,
                                freq='D')

transactions_list = []

for ts in TS_transactions_list:
            series = TimeSeries.from_series(ts.pd_series())   # necessary workaround to remove static covariates (so I can stack covariates later on)
            series = series.astype(np.float32)
            transactions_list.append(series)

# transactions_list[24] = transactions_list[24].slice(start_ts=pd.Timestamp('20130102'), end_ts=pd.Timestamp('20170815'))

from datetime import datetime, timedelta

transactions_list_full = []

for ts in transactions_list:
  if ts.start_time() > pd.Timestamp('20130101'):
    # end_time = (ts.start_time() - timedelta(days=1))
    # delta = end_time - pd.Timestamp('20130101')
    delta = ts.start_time() - pd.Timestamp('20130101')
    zero_series = TimeSeries.from_times_and_values(
                              times=pd.date_range(start=pd.Timestamp('20130101'),
                              # end=end_time, freq="D"),
                              end=ts.start_time() - timedelta(days=1), freq="D"),
                              # values=np.zeros(delta.days+1))
                              values=np.zeros(delta.days))
    ts = zero_series.append(ts)
    # transactions_list_full.append(ts)
  transactions_list_full.append(ts)

transactions_filler = MissingValuesFiller(verbose=False, n_jobs=-1, name="Filler")
transactions_scaler = Scaler(verbose=False, n_jobs=-1, name="Scaler")

transactions_pipeline = Pipeline([transactions_filler, transactions_scaler])
transactions_transformed = transactions_pipeline.fit_transform(transactions_list_full)

# Moving Averages for Transactions
trans_moving_average_7 = MovingAverage(window=7)
trans_moving_average_28 = MovingAverage(window=28)

transactions_covs = []

for ts in transactions_transformed:
  ma_7 = trans_moving_average_7.filter(ts).astype(np.float32)
  ma_7 = ma_7.with_columns_renamed(col_names=ma_7.components, col_names_new="transactions_ma_7")
  ma_28 = trans_moving_average_28.filter(ts).astype(np.float32)
  ma_28 = ma_28.with_columns_renamed(col_names=ma_28.components, col_names_new="transactions_ma_28")
  trans_and_mas = ts.with_columns_renamed(col_names=ts.components, col_names_new="transactions").stack(ma_7).stack(ma_28)
  transactions_covs.append(trans_and_mas)

# transactions_list_full[24]
transactions_covs[0]

<TimeSeries (DataArray) (time: 1688, component: 3, sample: 1)>
array([[[0.        ],
        [0.48023486],
        [0.48931998]],

       [[0.69831293],
        [0.4840225 ],
        [0.49374792]],

       [[0.60635131],
        [0.43202117],
        [0.50285316]],

       ...,

       [[0.13761164],
        [0.395854  ],
        [0.46284369]],

       [[0.57327158],
        [0.35831955],
        [0.48104119]],

       [[0.5600397 ],
        [0.40076083],
        [0.47182709]]])
Coordinates:
  * time       (time) datetime64[ns] 2013-01-01 2013-01-02 ... 2017-08-15
  * component  (component) object 'transactions' ... 'transactions_ma_28'
Dimensions without coordinates: sample
Attributes:
    static_covariates:  None
    hierarchy:          None

In [ ]:


# Re-Defining Categories of Holidays in a Meaningful Way

df_holidays_events['type'] = np.where(df_holidays_events['transferred'] == True,'Transferred', 
                                      df_holidays_events['type'])

df_holidays_events['type'] = np.where(df_holidays_events['type'] == 'Transfer','Holiday', 
                                      df_holidays_events['type'])

df_holidays_events['type'] = np.where(df_holidays_events['type'] == 'Additional','Holiday', 
                                      df_holidays_events['type'])

df_holidays_events['type'] = np.where(df_holidays_events['type'] == 'Bridge','Holiday', 
                                      df_holidays_events['type'])

holiday_df_per_store = []

for i in range(0,len(df_stores)):
  df_holiday_event_mask = pd.DataFrame(columns=['date'])
  df_holiday_event_mask["date"] = df_holidays_events["date"]

  df_holiday_event_mask["national_holiday"] = np.where(((df_holidays_events["type"] == "Holiday") & 
                                                      (df_holidays_events["locale"] == "National")), 1, 0)
  df_holiday_event_mask["earthquake_relief"] = np.where(df_holidays_events['description'].str.contains('Terremoto Manabi'), 1, 0)
  df_holiday_event_mask["christmas"] = np.where(df_holidays_events['description'].str.contains('Navidad'), 1, 0)
  df_holiday_event_mask["football_event"] = np.where(df_holidays_events['description'].str.contains('futbol'), 1, 0)
  df_holiday_event_mask["national_event"] = np.where(((df_holidays_events["type"] == "Event") & 
                                                    (df_holidays_events["locale"] == "National") & 
                                                    (~df_holidays_events['description'].str.contains('Terremoto Manabi')) & 
                                                    (~df_holidays_events['description'].str.contains('futbol'))), 1, 0)
  # no local event in dataset
  df_holiday_event_mask["work_day"] = np.where((df_holidays_events["type"] == "Work Day"), 1, 0)
  df_holiday_event_mask["regional_holiday"] = np.where((df_holidays_events["type"] == "Holiday") & 
                                                  (df_holidays_events["locale_name"] == df_stores['state'][i]), 1, 0)
  df_holiday_event_mask["local_holiday"] = np.where((df_holidays_events["type"] == "Holiday") &
                                                    (df_holidays_events["locale_name"] == df_stores['city'][i]), 1, 0)
#             df_holiday_dummies["local_holiday"] = np.where(((df_holidays_events["type"] == "Holiday") & 
#                                                             ((df_holidays_events["locale_name"] == df_stores['state'][i]) | 
#                                                              (df_holidays_events["locale_name"] == df_stores['city'][i]))), 1, 0)
            
  holiday_df_per_store.append(df_holiday_event_mask)


# remove all zeros mask and duplicates dates in holiday_list

holiday_df_filter_per_store = []

for i in range(0,len(holiday_df_per_store)):     
  df_holiday = holiday_df_per_store[i].set_index('date')
  df_holiday = df_holiday.loc[~((df_holiday==0).all(axis=1))]
  df_holiday = df_holiday.groupby('date').agg({'national_holiday':'max', 'earthquake_relief':'max', 
                          'christmas':'max', 'football_event':'max', 'regional_holiday': 'max', 'work_day':'max',
                          # 'christmas':'max', 'football_event':'max', 'work_day':'max',
                          'national_event':'max', 'local_holiday':'max'}).reset_index()
  holiday_df_filter_per_store.append(df_holiday)

# holiday_df_filter_per_store[24][-1:-20:-1][:]

holiday_list_store = []

df_wage = pd.DataFrame(index=pd.date_range(start=pd.Timestamp('20130101'),end=pd.Timestamp('20170831'),freq='D')
,columns=['workday','wage'])
df_wage['workday'] = np.where(df_wage.index.dayofweek < 5, 1, 0)
df_wage['wage'] = np.where((df_wage.index.day == 15) | (df_wage.index.is_month_end), 1, 0)
wage_TS = TimeSeries.from_dataframe(df_wage)
cols = ['national_holiday','regional_holiday','local_holiday']


for i in range(0,len(df_stores)): 
  holidays_TS = TimeSeries.from_dataframe(holiday_df_filter_per_store[i], 
                              time_col = 'date',
                              fill_missing_dates=True,
                              fillna_value=0,
                              freq='D')
  holidays_TS = holidays_TS.slice(pd.Timestamp('20130101'),pd.Timestamp('20170831'))
  holidays_TS = holidays_TS.stack(wage_TS)
  holidays_df = holidays_TS.pd_dataframe()
  holidays_df['workday'] = np.where((holidays_df[['work_day','workday']].max(1) == 1) & 
                                  (holidays_df[cols].max(1) == 0), 1, 0)
  holidays_df.drop('work_day',axis=1, inplace=True)
  holidays_TS = TimeSeries.from_dataframe(holidays_df)
  holidays_TS = holidays_TS.astype(np.float32)
  holiday_list_store.append(holidays_TS)

# holiday_list_store[24].pd_dataframe()[1400:1450]

holidays_filler = MissingValuesFiller(verbose=False, n_jobs=-1, name="Filler")
holidays_scaler = Scaler(verbose=False, n_jobs=-1, name="Scaler")

holidays_pipeline = Pipeline([holidays_filler, holidays_scaler])
holidays_transformed = holidays_pipeline.fit_transform(holiday_list_store)

holidays_transformed[24].pd_dataframe()[1400:1425]

component,national_holiday,earthquake_relief,christmas,football_event,regional_holiday,national_event,local_holiday,workday,wage
date,,,,,,,,,
2016-11-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2016-11-02,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2016-11-03,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2016-11-04,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2016-11-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2016-11-06,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2016-11-07,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2016-11-08,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2016-11-09,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [ ]:

# Stack Together Store-Specific Covariates with General Covariates

store_covariates_future = []

for store in range(0,len(store_list)):
  stacked_covariates = holidays_transformed[store].stack(general_covariates)  
  store_covariates_future.append(stacked_covariates)

store_covariates_past = []
holidays_transformed_sliced = holidays_transformed # for slicing past covariates

for store in range(0,len(store_list)):
  holidays_transformed_sliced[store] = holidays_transformed[store].slice_intersect(transactions_covs[store])
  general_covariates_sliced = general_covariates.slice_intersect(transactions_covs[store])
  stacked_covariates = transactions_covs[store].stack(holidays_transformed_sliced[store]).stack(general_covariates_sliced)  
  store_covariates_past.append(stacked_covariates)
    
# Store/Family-Varying Covariates (Promotion)

df_promotion = pd.concat([df_train, df_test], axis=0)
df_promotion = df_promotion.sort_values(["store_nbr","family","date"])
df_promotion.tail()

family_promotion_dict = {}

for family in family_list:
  df_family = df_promotion.loc[df_promotion['family'] == family]

  list_of_TS_promo = TimeSeries.from_group_dataframe(
                                df_family,
                                time_col="date",
                                group_cols=["store_nbr","family"],  # individual time series are extracted by grouping `df` by `group_cols`
                                value_cols="onpromotion", # covariate of interest
                                fill_missing_dates=True,
                                freq='D')
  
  for ts in list_of_TS_promo:
            ts = ts.astype(np.float32)

  family_promotion_dict[family] = list_of_TS_promo

promotion_transformed_dict = {}

for key in tqdm(family_promotion_dict):
  promo_filler = MissingValuesFiller(verbose=False, n_jobs=-1, name="Fill NAs")
  promo_scaler = Scaler(verbose=False, n_jobs=-1, name="Scaling")

  promo_pipeline = Pipeline([promo_filler,
                             promo_scaler])
  
  promotion_transformed = promo_pipeline.fit_transform(family_promotion_dict[key])

  # Moving Averages for Promotion Family Dictionaries
  promo_moving_average_7 = MovingAverage(window=7)
  promo_moving_average_28 = MovingAverage(window=28)

  promotion_covs = []

  for ts in promotion_transformed:
    ma_7 = promo_moving_average_7.filter(ts)
    ma_7 = TimeSeries.from_series(ma_7.pd_series())  
    ma_7 = ma_7.astype(np.float32)
    ma_7 = ma_7.with_columns_renamed(col_names=ma_7.components, col_names_new="promotion_ma_7")
    ma_28 = promo_moving_average_28.filter(ts)
    ma_28 = TimeSeries.from_series(ma_28.pd_series())  
    ma_28 = ma_28.astype(np.float32)
    ma_28 = ma_28.with_columns_renamed(col_names=ma_28.components, col_names_new="promotion_ma_28")
    promo_and_mas = ts.stack(ma_7).stack(ma_28)
    promotion_covs.append(promo_and_mas)

  promotion_transformed_dict[key] = promotion_covs

# 2.5. Assemble All Covariates in Dictionaries

past_covariates_dict = {}

for key in tqdm(promotion_transformed_dict):

  promotion_family = promotion_transformed_dict[key]
  sales_mas = sales_moving_averages_dict[key]
  covariates_past = [promotion_family[i].slice_intersect(store_covariates_past[i]).stack(store_covariates_past[i].stack(sales_mas[i])) for i in range(0,len(promotion_family))]

  past_covariates_dict[key] = covariates_past

future_covariates_dict = {}

for key in tqdm(promotion_transformed_dict):

  promotion_family = promotion_transformed_dict[key]
  covariates_future = [promotion_family[i].stack(store_covariates_future[i]) for i in range(0,len(promotion_family))]

  future_covariates_dict[key] = covariates_future

# only_past_covariates_dict = {}

# for key in tqdm(sales_moving_averages_dict):
#   sales_moving_averages = sales_moving_averages_dict[key]
#   only_past_covariates = [sales_moving_averages[i].stack(transactions_covs[i]) for i in range(0,len(sales_moving_averages))]

#   only_past_covariates_dict[key] = only_past_covariates

# Delete Original Dataframes to Save Memory

del(df_train)
del(df_test)
del(df_stores)
del(df_holidays_events)
del(df_oil)
del(df_transactions)
gc.collect()

100%|██████████| 33/33 [00:07<00:00,  4.17it/s]


6223

2. Train Global LightGDM Model

In [ ]:

# Train 33 Global LightGBM Models (one for each product family)

from sklearn.metrics import mean_squared_log_error as msle, mean_squared_error as mse
from lightgbm import early_stopping

#Kenny - Use Optuna to help tune LightGBM (IN PROGRESS)
LGBM_Forecasts_Families = {}
LGBM_Forecasts_Families_back = {}
forecast_list_LGBM = []
sales_data = []
LGBM_rmsle = []

def flatten(l):
  return [item for item in l]

def build_lightgdm(lags,
                   lags_future_covariates,
                   lags_past_covariates,
                   sales_family,
                   training_data,
                   TCN_covariates,
                   PAST_covariates,
                   train_sliced):
  
  LGBM_Model = LightGBMModel(lags=lags,
                             lags_future_covariates=lags_future_covariates,
                             lags_past_covariates=lags_past_covariates,
                             output_chunk_length=1,
                             random_state=2022,
                             gpu_use_dp= "false")
  
  LGBM_Model.fit(series=train_sliced, 
                 future_covariates=TCN_covariates,
                 past_covariates=PAST_covariates,
                 verbose=True)
  
  return LGBM_Model

def objective(trial, family):
  # lags = trial.suggest_categorical("lags", [30, 60, 62])
  # lags_future_covariates = trial.suggest_categorical("lags_future_covariates", [(14, 1), (16,1), (32,1)])
  # lags_past_covariates = trial.suggest_categorical("lags_past_covariates", [[-16,-17,-18,-19,-20,-21,-22,-23,-24,-25,-26,-27,-28,-29],
  #                                                                           [-16,-17,-18,-19,-20,-21,-22,-23,-24,-25,-26,-27,-28,-29,-30,-32]])
  lags = trial.suggest_categorical("lags", [60, 62])
  lags_future_covariates = trial.suggest_categorical("lags_future_covariates", [(14, 1), (16,1)])
  lags_past_covariates = trial.suggest_categorical("lags_past_covariates", [[-16, -18]])
  
  sales_family = family_TS_transformed_dict[family]
  training_data = [ts[:-16] for ts in sales_family]
  TCN_covariates = future_covariates_dict[family]
  PAST_covariates = past_covariates_dict[family]
  train_sliced = [training_data[i].slice_intersect(TCN_covariates[i]) for i in range(0,len(training_data))]

  model = build_lightgdm(lags=lags,
                         lags_future_covariates=lags_future_covariates,
                         lags_past_covariates=lags_past_covariates,
                         sales_family=sales_family,
                         training_data=training_data,
                         TCN_covariates=TCN_covariates,
                         PAST_covariates=PAST_covariates,
                         train_sliced=train_sliced
                         )

  forecast_LGBM = model.predict(n=16,
                                series=train_sliced,
                                future_covariates=TCN_covariates,
                                past_covariates=PAST_covariates)
  
  #update
  LGBM_Forecasts_Families[family] = forecast_LGBM
  LGBM_Forecasts_Families_back[family] = family_pipeline_dict[family].inverse_transform(LGBM_Forecasts_Families[family], partial=True)

  #zero forecasting
  for n in range(0,len(LGBM_Forecasts_Families_back[family])):
    if (family_TS_dict[family][n][:-16].univariate_values()[-21:] == 0).all():
        LGBM_Forecasts_Families_back[family][n] = LGBM_Forecasts_Families_back[family][n].map(lambda x: x * 0)
  
  # Building all 1782 Forecasts in one List - Kenny: To be used later?
  forecast_list_LGBM.append(LGBM_Forecasts_Families_back[family])
  sales_data.append(family_TS_dict[family])
  
  # Evaluate performance for each family of forecasts
  actual_list = flatten(family_TS_dict[family])
  pred_list_LGBM = flatten(LGBM_Forecasts_Families_back[family])

  # Return score 
  err = rmsle(actual_series = actual_list,
                 pred_series = pred_list_LGBM,
                 n_jobs = -1)
  
  err_val = np.mean(err)

  return err_val

def print_callback(study, trial):
  LGBM_rmsle.append(trial.value)
  print(f"Current value: {trial.value}, Current params: {trial.params}")

100%|██████████| 33/33 [32:00<00:00, 58.21s/it]


In [ ]:
#Actual training
for family in tqdm(family_list):

  study_lightgbm = optuna.create_study(direction="minimize")
  study_lightgbm.optimize(lambda trial: objective(trial, family), n_trials=5, callbacks=[print_callback])

# Best parameters
print(f"Best value: {study_lightgbm.best_value}, Best params: {study_lightgbm.best_trial.params}")

# Avg error for all families
LGBM_rmsle_val = np.mean(LGBM_rmsle)
print("\n")
print("The mean RMSLE for the 33 LightGBM Global Product Family Models over all 1782 series is {:.5f}.".format(LGBM_rmsle_val))
print("\n")

In [ ]:
# Re-Format all 1782 Forecasts in one List and Evaluate Performance

# Mean RMSLE for Families

family_forecast_rmsle_LGBM = {}

for family in family_list:

  LGBM_rmsle_family = rmsle(actual_series = family_TS_dict[family],
                 pred_series = LGBM_Forecasts_Families_back[family],
                 n_jobs = -1,
                 inter_reduction=np.mean)
  
  family_forecast_rmsle_LGBM[family] = LGBM_rmsle_family

family_forecast_rmsle_LGBM = dict(sorted(family_forecast_rmsle_LGBM.items(), key=lambda item: item[1]))

print("Mean RMSLE for the 33 different product families, from worst to best:")
print("\n")

# Iterate over key/value pairs in dict and print them
for key, value in family_forecast_rmsle_LGBM.items():
    print(key, ' : ', value)



The mean RMSLE for the 33 LightGBM Global Product Family Models over all 1782 series is 0.33494.


Mean RMSLE for the 33 different product families, from worst to best:


BOOKS  :  0.049800587298245755
DAIRY  :  0.13518545028485632
PRODUCE  :  0.1404670807975459
GROCERY I  :  0.15171929376016366
BREAD/BAKERY  :  0.1650185409717471
DELI  :  0.1729640669936199
MEATS  :  0.19493427174106456
PERSONAL CARE  :  0.20105218076462297
BABY CARE  :  0.20168810275429236
POULTRY  :  0.20404640751490993
HOME CARE  :  0.2078176556668312
BEVERAGES  :  0.22016755909604585
EGGS  :  0.2554540257411625
PREPARED FOODS  :  0.2613807075068013
FROZEN FOODS  :  0.26615755553008824
CLEANING  :  0.27393699806065724
HOME APPLIANCES  :  0.28833222473364784
LAWN AND GARDEN  :  0.3527046933529629
LIQUOR,WINE,BEER  :  0.4114532472788173
HOME AND KITCHEN II  :  0.4282772735571317
LADIESWEAR  :  0.43242884158684175
PLAYERS AND ELECTRONICS  :  0.44181496002686155
PET SUPPLIES  :  0.4497637312161891
SEAFOOD  :  0.45194

4. Train Global LightGDM Model with Full Training Data

In [ ]:
# Train 33 Global LightGBM Models (one for each product family)

from sklearn.metrics import mean_squared_log_error as msle, mean_squared_error as mse
from lightgbm import early_stopping

# Train 33 Global LightGBM Models with Full Data

LGBM_Models_Submission = {}

for family in tqdm(family_list):

  # Define Data for family
  sales_family = family_TS_transformed_dict[family]
  training_data = [ts for ts in sales_family] 
  TCN_covariates = future_covariates_dict[family]
  train_sliced = [training_data[i].slice_intersect(TCN_covariates[i]) for i in range(0,len(training_data))]

  LGBM_Model_Submission = LightGBMModel(lags = 63,
                             lags_future_covariates = (14,1),
                            #  lags_past_covariates = [-16,-17,-18,-19,-20,-21,-22],
                            #  lags_past_covariates = [-16,-17,-18,-19,-20,-21,-22,-23,-24,-25,-26,-27,-28,-29],
                             lags_past_covariates = [ i for i in range(-16, -30, -1)],
                             output_chunk_length=1,
                             random_state=2022,
                         #    max_bin= [63],
                             gpu_use_dp= "false")
     
  LGBM_Model_Submission.fit(series=train_sliced, 
                        future_covariates=TCN_covariates,
                        # past_covariates=transactions_transformed,
                        past_covariates=past_covariates_dict[family],
                        verbose=True)

  LGBM_Models_Submission[family] = LGBM_Model_Submission

 91%|█████████ | 30/33 [27:40<02:41, 53.71s/it]

5. Forecast Test Data for Submission

In [ ]:
# Generate Forecasts for Submission

LGBM_Forecasts_Families_Submission = {}

for family in tqdm(family_list):

  sales_family = family_TS_transformed_dict[family]
  training_data = [ts for ts in sales_family]
  LGBM_covariates = future_covariates_dict[family]
  train_sliced = [training_data[i].slice_intersect(TCN_covariates[i]) for i in range(0,len(training_data))]

  forecast_LGBM = LGBM_Models_Submission[family].predict(n=16,
                                         series=train_sliced,
                                         future_covariates=LGBM_covariates,
                                        #  past_covariates=transactions_transformed)
                                        past_covariates=past_covariates_dict[family])
  
  LGBM_Forecasts_Families_Submission[family] = forecast_LGBM

# Transform Back

LGBM_Forecasts_Families_back_Submission = {}

for family in tqdm(family_list):

  LGBM_Forecasts_Families_back_Submission[family] = family_pipeline_dict[family].inverse_transform(LGBM_Forecasts_Families_Submission[family], partial=True)

# Zero Forecasting

for family in tqdm(LGBM_Forecasts_Families_back_Submission):
  for n in range(0,len(LGBM_Forecasts_Families_back_Submission[family])):
    if (family_TS_dict[family][n].univariate_values()[-21:] == 0).all():
        LGBM_Forecasts_Families_back_Submission[family][n] = LGBM_Forecasts_Families_back_Submission[family][n].map(lambda x: x * 0)
        
# Prepare Submission in Correct Format

listofseries = []

for store in range(0,54):
  for family in tqdm(family_list):
      oneforecast = LGBM_Forecasts_Families_back_Submission[family][store].pd_dataframe()
      oneforecast.columns = ['fcast']
      listofseries.append(oneforecast)

df_forecasts = pd.concat(listofseries) 
df_forecasts.reset_index(drop=True, inplace=True)

# No Negative Forecasts
df_forecasts[df_forecasts < 0] = 0
forecasts_kaggle = pd.concat([df_test_sorted, df_forecasts.set_index(df_test_sorted.index)], axis=1)
forecasts_kaggle_sorted = forecasts_kaggle.sort_values(by=['id'])
forecasts_kaggle_sorted = forecasts_kaggle_sorted.drop(['date','store_nbr','family'], axis=1)
forecasts_kaggle_sorted = forecasts_kaggle_sorted.rename(columns={"fcast": "sales"})
forecasts_kaggle_sorted = forecasts_kaggle_sorted.reset_index(drop=True)

# Submission
submission_kaggle = forecasts_kaggle_sorted
submission_kaggle.to_csv('/content/drive/My Drive/Colab Notebooks/submission.csv', index=False)

100%|██████████| 33/33 [00:00<00:00, 1412.56it/s]
